# 📊 Atividade — Visualização de Dados após ETL
## TechStore · Do dado bruto ao dashboard

**Contexto:** Você acabou de executar um pipeline ETL completo na empresa **TechStore**.  
Os dados de clientes, pedidos e produtos foram integrados, limpos e carregados no Data Warehouse.  
Agora é hora de transformar esses dados em **visualizações que contem uma história**.

---

### 🗺️ Roteiro da Atividade

| Etapa | Descrição | Tipo |
|-------|-----------|------|
| 0 | Setup e dados | ✅ Execute sem modificar |
| 1 | Gráfico de barras — Receita por categoria | 🟡 Complete o código |
| 2 | Gráfico de linha — Evolução mensal | 🟡 Complete o código |
| 3 | Gráfico de pizza — Distribuição por segmento | 🟡 Complete o código |
| 4 | Heatmap — Vendas por estado e mês | 🟡 Complete o código |
| 5 | Dashboard completo | 🟡 Combine tudo em uma figura |
| 6 | 🏆 Desafio Livre | Você decide qual gráfico criar |

---

### 📋 Regras
- 🟡 `# TODO:` → complete o código
- 🟢 `# DICA:` → use se travar por mais de 3 minutos
- Execute **sempre em ordem** — cada célula depende da anterior
- No desafio final: **sem dicas** e sem gabarito!

> 💡 **Meta:** ao final, você terá um painel de análise completo pronto para apresentar a um gestor.


---
## 🔧 Etapa 0 — Setup e Dados
> Execute sem modificar. Os dados simulam o resultado de um pipeline ETL completo.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

# Estilo visual padrão
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8FAFC',
    'axes.grid':        True,
    'grid.color':       '#E2E8F0',
    'grid.linewidth':   0.7,
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

PALETA = ['#2563EB','#0D9488','#D97706','#16A34A','#DC2626','#7C3AED','#EC4899','#0EA5E9']

print("✅ Bibliotecas carregadas!")
print(f"   matplotlib : {plt.matplotlib.__version__}")
print(f"   seaborn    : {sns.__version__}")
print(f"   pandas     : {pd.__version__}")


In [ ]:
# ── DADOS DO DATA WAREHOUSE (resultado do ETL) ──────────────────────────────
random.seed(42)
np.random.seed(42)

categorias  = ['Computadores','Periféricos','Monitores','Armazenamento','Áudio','Acessórios']
estados     = ['SP','RJ','MG','RS','PR','BA','CE','GO','SC','PE']
segmentos   = ['Varejo','Corporativo']
produtos_cat= {
    'Computadores':  ['Notebook Pro 15','Ultrabook Slim','Desktop Gamer'],
    'Periféricos':   ['Mouse Sem Fio','Teclado Mecânico','Webcam Full HD'],
    'Monitores':     ["Monitor 24''","Monitor 27'' 4K"],
    'Armazenamento': ['SSD 1TB','HD Externo 2TB'],
    'Áudio':         ['Headset USB','Caixa Bluetooth'],
    'Acessórios':    ['Hub USB-C','Suporte Notebook'],
}
custo_pct = {'Computadores':0.60,'Periféricos':0.42,'Monitores':0.52,
             'Armazenamento':0.46,'Áudio':0.38,'Acessórios':0.41}

registros = []
base = datetime(2024, 1, 1)
for i in range(800):
    cat     = random.choice(categorias)
    prod    = random.choice(produtos_cat[cat])
    seg     = random.choices(segmentos, weights=[65,35])[0]
    estado  = random.choice(estados)
    data    = base + timedelta(days=random.randint(0, 364))
    preco   = round(random.uniform(80, 3500) * (1.8 if cat=='Computadores' else 1.0), 2)
    qtd     = random.randint(1, 5 if seg=='Varejo' else 15)
    receita = round(preco * qtd, 2)
    custo   = round(receita * custo_pct[cat], 2)
    status  = random.choices(['concluido','cancelado','em_andamento'], weights=[75,12,13])[0]
    registros.append({
        'pedido_id':  i+1,
        'produto':    prod,
        'categoria':  cat,
        'segmento':   seg,
        'estado':     estado,
        'data':       data,
        'mes':        data.month,
        'mes_nome':   data.strftime('%b'),
        'trimestre':  f"T{(data.month-1)//3+1}",
        'preco_unit': preco,
        'quantidade': qtd,
        'receita':    receita,
        'custo':      custo,
        'lucro':      round(receita - custo, 2),
        'status':     status,
    })

df_dw = pd.DataFrame(registros)
df_ok = df_dw[df_dw['status'] == 'concluido'].copy()

print(f"✅ Data Warehouse carregado!")
print(f"   Total de pedidos    : {len(df_dw)}")
print(f"   Pedidos concluídos  : {len(df_ok)}")
print(f"   Período             : Jan 2024 → Dez 2024")
print(f"   Receita total       : R$ {df_ok['receita'].sum():,.2f}")
print(f"   Lucro total         : R$ {df_ok['lucro'].sum():,.2f}")
print()
print("Primeiras linhas do DW:")
df_ok[['pedido_id','produto','categoria','segmento','estado','receita','lucro','status']].head(5)


---
## 📊 Atividade 1 — Gráfico de Barras: Receita por Categoria
### 🟡 Nível 1 — Complete as lacunas

**Objetivo:** visualizar qual categoria de produto gera mais receita.

**Pergunte antes de codar:** Qual agregação transforma linhas em totais por grupo?


In [ ]:
# PASSO 1: Agregar dados — receita e lucro por categoria
receita_cat = (
    df_ok
    .groupby(___)            # TODO: agrupar por qual coluna?
    .agg(
        receita = ('receita', ___),   # TODO: qual função para somar?
        lucro   = ('lucro',   'sum'),
        pedidos = ('pedido_id', 'count')
    )
    .sort_values(___, ascending=False)   # TODO: ordenar pela maior receita
    .reset_index()
)

print("Dados agregados:")
print(receita_cat)


> 🟢 **DICA 1:** `.groupby('categoria')` | agregação: `'sum'` | ordenar por: `'receita'`

In [ ]:
# PASSO 2: Criar o gráfico de barras
fig, ax = plt.subplots(figsize=(10, 5))

# TODO: crie as barras horizontais com ax.barh()
# barras de receita
bars = ax.barh(
    receita_cat[___],         # TODO: qual coluna vai no eixo Y (categorias)?
    receita_cat[___],         # TODO: qual coluna vai no eixo X (valores)?
    color=PALETA[:len(receita_cat)],
    height=0.55,
    edgecolor='white',
    linewidth=0.8
)

# Rótulos de valor dentro de cada barra
for bar in bars:
    w = bar.get_width()
    ax.text(w * 0.97, bar.get_y() + bar.get_height()/2,
            f'R$ {w:,.0f}', va='center', ha='right',
            color='white', fontsize=9, fontweight='bold')

# TODO: complete os rótulos do gráfico
ax.set_title(___, fontsize=14, fontweight='bold', pad=12)  # TODO: título descritivo
ax.set_xlabel(___)                                          # TODO: rótulo do eixo X
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R$ {x/1000:.0f}k'))
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('grafico_01_receita_categoria.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico 1 salvo!")


> 🟢 **DICA 2:**
> - `ax.barh(receita_cat['categoria'], receita_cat['receita'], ...)`
> - Título sugerido: `'Receita Total por Categoria de Produto — 2024'`
> - Label X: `'Receita (R$)'`


---
## 📈 Atividade 2 — Gráfico de Linha: Evolução Mensal da Receita
### 🟡 Nível 1 — Complete as lacunas

**Objetivo:** mostrar como a receita evolui mês a mês ao longo de 2024.

**Pense:** para plotar uma linha, os dados precisam estar em qual ordem?


In [ ]:
# PASSO 1: Agregar por mês
mensal = (
    df_ok
    .groupby([___, ___])     # TODO: agrupar por 'mes' E 'mes_nome' juntos
    .agg(
        receita = ('receita', 'sum'),
        lucro   = ('lucro',   'sum'),
        pedidos = ('pedido_id','count')
    )
    .sort_values(___)        # TODO: ordenar pelo número do mês
    .reset_index()
)
print(mensal[['mes','mes_nome','receita','lucro','pedidos']].to_string(index=False))


> 🟢 **DICA:** `groupby(['mes','mes_nome'])` | ordenar por: `'mes'`

In [ ]:
# PASSO 2: Gráfico de linha com área preenchida
fig, ax = plt.subplots(figsize=(12, 5))

# TODO: plote a linha de receita
ax.plot(
    mensal[___],     # TODO: eixo X — nome do mês
    mensal[___],     # TODO: eixo Y — receita
    color=PALETA[0],
    linewidth=2.5,
    marker='o',
    markersize=7,
    label='Receita'
)

# Área preenchida sob a curva de receita
ax.fill_between(mensal[___], mensal[___],   # TODO: mesmos argumentos do plot
                alpha=0.12, color=PALETA[0])

# TODO: adicione também a linha de lucro (color=PALETA[1], label='Lucro')
ax.plot(___, ___, color=PALETA[1], linewidth=2, marker='s',
        markersize=6, linestyle='--', label='Lucro')

# Anotações de valor nos picos
for _, row in mensal.iterrows():
    ax.annotate(f"R${row['receita']/1000:.0f}k",
                xy=(row['mes_nome'], row['receita']),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=7.5, color=PALETA[0])

ax.set_title(___, fontsize=14, fontweight='bold', pad=12)  # TODO: título
ax.set_xlabel('Mês')
ax.set_ylabel('Valor (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1000:.0f}k'))
ax.legend(framealpha=0.9)

plt.tight_layout()
plt.savefig('grafico_02_evolucao_mensal.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico 2 salvo!")


> 🟢 **DICA:**
> - `ax.plot(mensal['mes_nome'], mensal['receita'], ...)`
> - `ax.fill_between(mensal['mes_nome'], mensal['receita'], ...)`
> - Linha de lucro: `ax.plot(mensal['mes_nome'], mensal['lucro'], ...)`
> - Título: `'Evolução Mensal de Receita e Lucro — 2024'`


---
## 🥧 Atividade 3 — Gráfico de Pizza: Receita por Segmento
### 🟡 Nível 2 — Menos scaffolding

**Objetivo:** mostrar a proporção de receita entre clientes Varejo e Corporativo.

**Pense:** quando usar pizza? Quando faz mais sentido usar barras?


In [ ]:
# PASSO 1: Agregar por segmento
seg_dados = (
    df_ok
    .groupby('segmento')
    .agg(receita=('receita','sum'), pedidos=('pedido_id','count'), lucro=('lucro','sum'))
    .reset_index()
)

# PASSO 2: Gráfico de pizza com anel (donut)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Pizza de RECEITA ---
ax1 = axes[0]
wedge_props = dict(width=0.55, edgecolor='white', linewidth=2.5)

# TODO: crie o gráfico de pizza/donut
wedges, texts, autotexts = ax1.pie(
    seg_dados[___],          # TODO: qual coluna com os valores?
    labels=seg_dados[___],   # TODO: qual coluna com os rótulos?
    autopct=___,             # TODO: formato de porcentagem: '%1.1f%%'
    colors=PALETA[:2],
    startangle=90,
    wedgeprops=wedge_props,
    pctdistance=0.75
)
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight('bold'); at.set_color('white')

# Valor total no centro do donut
total = seg_dados['receita'].sum()
label_centro = f'R$ {"{:.0f}".format(total/1000)}k\ntotal'
ax1.text(0, 0, label_centro, ha='center', va='center',
         fontsize=12, fontweight='bold', color='#1E293B')
ax1.set_title('Receita por Segmento', fontsize=13, fontweight='bold', pad=15)

# --- Pizza de PEDIDOS ---
ax2 = axes[1]
# TODO: repita a lógica para a coluna 'pedidos'
wedges2, texts2, autotexts2 = ax2.pie(
    ___,                     # TODO: coluna de pedidos
    labels=seg_dados[___],   # TODO: coluna de rótulos
    autopct='%1.1f%%',
    colors=[PALETA[2], PALETA[3]],
    startangle=90,
    wedgeprops=wedge_props,
    pctdistance=0.75
)
for at in autotexts2:
    at.set_fontsize(11); at.set_fontweight('bold'); at.set_color('white')

total_ped = seg_dados['pedidos'].sum()
label_ped = str(total_ped) + chr(10) + 'pedidos'
ax2.text(0, 0, label_ped, ha='center', va='center',
         fontsize=12, fontweight='bold', color='#1E293B')
ax2.set_title('Pedidos por Segmento', fontsize=13, fontweight='bold', pad=15)

fig.suptitle('Distribuição por Segmento de Cliente — 2024',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafico_03_segmentos.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico 3 salvo!")


> 🟢 **DICA:**
> - `seg_dados['receita']` e `seg_dados['segmento']`
> - `autopct='%1.1f%%'`
> - Para pedidos: `seg_dados['pedidos']`


---
## 🗺️ Atividade 4 — Heatmap: Vendas por Estado e Trimestre
### 🟡 Nível 2 — Construa a tabela pivot

**Objetivo:** identificar padrões de sazonalidade por região.

**Pense:** para criar um heatmap, precisamos de uma tabela com linhas × colunas. Como transformar dados longos em largos?


In [ ]:
# PASSO 1: Criar tabela pivot — estados (linhas) × trimestres (colunas)
pivot_heat = df_ok.pivot_table(
    values=___,           # TODO: qual métrica plotar? (receita, lucro ou pedidos)
    index=___,            # TODO: qual coluna vira linha? (estado)
    columns=___,          # TODO: qual coluna vira coluna? (trimestre)
    aggfunc=___           # TODO: qual função de agregação? ('sum' ou 'mean')
).fillna(0)

print("Tabela pivot criada:")
print(pivot_heat.round(0))


> 🟢 **DICA:** `values='receita'` | `index='estado'` | `columns='trimestre'` | `aggfunc='sum'`

In [ ]:
# PASSO 2: Plotar o heatmap
fig, ax = plt.subplots(figsize=(9, 7))

# TODO: use sns.heatmap() para criar o heatmap
sns.heatmap(
    ___,                    # TODO: a tabela pivot criada acima
    annot=True,             # mostrar valores nas células
    fmt=___,                # TODO: formato dos números: '.0f' para inteiros
    cmap=___,               # TODO: paleta de cores: 'Blues', 'YlOrRd' ou 'Greens'
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Receita (R$)'}
)

ax.set_title(___, fontsize=14, fontweight='bold', pad=15)  # TODO: título
ax.set_xlabel('Trimestre', fontsize=11)
ax.set_ylabel('Estado', fontsize=11)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('grafico_04_heatmap_estados.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico 4 salvo!")


> 🟢 **DICA:**
> - `sns.heatmap(pivot_heat, ...)`
> - `fmt='.0f'`
> - `cmap='YlOrRd'` (amarelo → vermelho, quanto maior mais quente)
> - Título: `'Receita por Estado e Trimestre — 2024'`


---
## 📊 Atividade 5 — Barras Agrupadas: Receita vs Lucro por Categoria
### 🟡 Nível 2 — Posicione as barras manualmente

**Objetivo:** comparar receita e lucro lado a lado por categoria.

**Pense:** para barras agrupadas, como você calcula a posição de cada grupo no eixo X?


In [ ]:
# Dados já calculados na Atividade 1 — receita_cat
# Calcular margem percentual
receita_cat['margem_pct'] = (receita_cat['lucro'] / receita_cat['receita'] * 100).round(1)

# PASSO 1: Posições das barras
x = np.arange(len(receita_cat))
largura = 0.38

fig, ax = plt.subplots(figsize=(12, 5))

# TODO: crie duas séries de barras lado a lado
bars1 = ax.bar(
    x - ___,              # TODO: deslocar para a esquerda (x - largura/2)
    receita_cat[___],     # TODO: valores de receita
    width=largura,
    label='Receita',
    color=PALETA[0],
    alpha=0.9,
    edgecolor='white'
)
bars2 = ax.bar(
    x + ___,              # TODO: deslocar para a direita (x + largura/2)
    receita_cat[___],     # TODO: valores de lucro
    width=largura,
    label='Lucro',
    color=PALETA[2],
    alpha=0.9,
    edgecolor='white'
)

# Anotação de margem sobre cada par
for i, row in receita_cat.iterrows():
    ax.text(i, row['receita'] + 2000, f"{row['margem_pct']}%",
            ha='center', fontsize=8.5, color=PALETA[2], fontweight='bold')

# TODO: defina os ticks do eixo X com os nomes das categorias
ax.set_xticks(___)                    # TODO: posições x
ax.set_xticklabels(receita_cat[___],  # TODO: nome das categorias
                   rotation=15, ha='right')

ax.set_title('Receita vs Lucro por Categoria (% = margem)', fontsize=13, fontweight='bold')
ax.set_ylabel('Valor (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R${v/1000:.0f}k'))
ax.legend(framealpha=0.9)

plt.tight_layout()
plt.savefig('grafico_05_receita_lucro.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico 5 salvo!")


> 🟢 **DICA:**
> - `x - largura/2` e `x + largura/2`
> - `receita_cat['receita']` e `receita_cat['lucro']`
> - `ax.set_xticks(x)` e `ax.set_xticklabels(receita_cat['categoria'], ...)`


---
## 🖥️ Atividade 6 — Dashboard Completo
### 🟡 Nível 3 — Sem scaffolding de layout

**Objetivo:** combinar 4 visualizações em um único painel executivo.

**Pense antes de começar:**
- Quantas linhas e colunas de subplots você precisa?
- Como fazer um subplot ocupar mais de uma coluna?
- Qual gráfico merece mais destaque (mais espaço)?


In [ ]:
# TODO: monte o dashboard com plt.subplots() usando gridspec_kw ou subplot_mosaic
# Sugestão de layout:
#   ┌──────────────────┬───────────┐
#   │  Linha (mensal)  │  Pizza    │
#   ├────────┬─────────┴───────────┤
#   │ Barras │     Heatmap         │
#   └────────┴─────────────────────┘

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Dashboard TechStore — Análise de Vendas 2024',
             fontsize=16, fontweight='bold', y=1.01)

# TODO: crie os subplots e reproduza os 4 gráficos anteriores
# Exemplo de estrutura:
# ax1 = fig.add_subplot(2, 2, 1)  → linha superior esquerda
# ax2 = fig.add_subplot(2, 2, 2)  → linha superior direita
# ax3 = fig.add_subplot(2, 2, 3)  → linha inferior esquerda
# ax4 = fig.add_subplot(2, 2, 4)  → linha inferior direita

# Complete aqui:


plt.tight_layout()
plt.savefig('dashboard_techstore.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard salvo!")


> 🟢 **DICA — estrutura básica:**
> ```python
> fig, axes = plt.subplots(2, 2, figsize=(16, 10))
> ax1, ax2 = axes[0]   # linha 0: esquerda e direita
> ax3, ax4 = axes[1]   # linha 1: esquerda e direita
> ```
> Para subplot que ocupa 2 colunas: use `gridspec_kw={'width_ratios':[1,1]}`  
> ou `subplot_mosaic([['A','B'],['C','C']])` para layouts assimétricos.


---
## 🏆 Desafio Livre — Sua Análise
### Sem dicas · Sem gabarito · Criatividade total

Escolha UMA das análises abaixo (ou proponha a sua):

| # | Desafio | Tipo de gráfico sugerido |
|---|---------|--------------------------|
| A | **Top 10 produtos** por receita total | Barras horizontais |
| B | **Proporção de status** de pedidos (concluído, cancelado, em andamento) | Pizza ou barras empilhadas |
| C | **Dispersão** entre quantidade vendida e receita por produto | Scatter plot |
| D | **Receita acumulada** ao longo do ano (linha crescente) | Linha com área |
| E | **Comparação** de margem de lucro entre categorias | Barras com linha de média |
| F | **Você decide** — use os dados do DW para responder uma pergunta de negócio | Livre |

> ⚠️ Regra do desafio: o gráfico deve ter **título claro**, **eixos rotulados** e **cores com significado** (não apenas decorativas).


In [ ]:
# 🏆 DESAFIO LIVRE — Sua implementação aqui

# Dica: explore o DataFrame df_ok com:
# df_ok.columns          → todas as colunas disponíveis
# df_ok.describe()       → estatísticas gerais
# df_ok['coluna'].value_counts()  → contagem por valor

# Exemplo de agrupamento para começar:
# df_ok.groupby('produto').agg(receita=('receita','sum')).sort_values('receita',ascending=False).head(10)

# SEU CÓDIGO:


In [ ]:
# Verificação final — todos os gráficos gerados
import os
graficos = [
    ('grafico_01_receita_categoria.png', 'Receita por Categoria'),
    ('grafico_02_evolucao_mensal.png',   'Evolução Mensal'),
    ('grafico_03_segmentos.png',         'Segmentos'),
    ('grafico_04_heatmap_estados.png',   'Heatmap Estados'),
    ('grafico_05_receita_lucro.png',     'Receita vs Lucro'),
    ('dashboard_techstore.png',          'Dashboard Completo'),
]

print("=" * 50)
print("  RELATÓRIO FINAL DA ATIVIDADE")
print("=" * 50)
for nome, desc in graficos:
    existe = os.path.exists(nome)
    status = "✅" if existe else "❌ ainda não gerado"
    print(f"  {status}  {desc} ({nome})")

concluidos = sum(1 for n,_ in graficos if os.path.exists(n))
print()
print(f"  Gráficos concluídos: {concluidos}/{len(graficos)}")
if concluidos == len(graficos):
    print("  🎉 Atividade completa! Dashboard pronto para apresentar.")
else:
    print("  ⚠️  Complete as etapas pendentes.")
